# Create Beijing Municipal Science and Technology Commission Awards

Creates awards from the Beijing Municipal Science and Technology Commission's
(市科委、中关村管委会) quarterly "项目(课题)立项公开清单" -- its own disclosure of
newly approved plan-project 课题 on kw.beijing.gov.cn (live 2022 Q2 - 2023 Q1; older
quarters from the retired Hanweb site via Internet Archive raw captures).

**Prerequisites:** run `scripts/local/beijing_bmstc_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/beijing_bmstc/beijing_bmstc_projects.parquet`.

**Routing (how-to §2.3.2):** the portal is shared with the Beijing NSF; 北京市自然科学基金
rosters are NOT in this ingest (they go to `beijing_nsf`, priority 523).

**Filter:** research 课题 only. Programme lines that are financial-service / subsidy
money (科技金融, 补贴, 保险) are dropped in the script.

**funder_award_id (§2.1.1):** the lists print no 课题 number (papers cite Z-numbers
such as Z181100001918023 that are not published anywhere public), so each 课题 gets
the stable synthetic key `BMSTC-<sha1(课题名称|承担单位)[:12]>` -- it collapses onto 0
existing citation stubs.

**Amounts:** 市财政经费 (万元) converted to CNY in the script. **Dates:** 课题实施周期.

**PI names:** Chinese, family-first -> `family_name`, `given_name` NULL.

**Funder details (Path A, F4320*):**
- funder_id: `4320325902` (Beijing Municipal Science and Technology Commission, CN)
- related programme row `4320335843` "Beijing Science and Technology Planning Project" -- flagged, not used.

**Priority:** `510` (direct funder ingest; higher wins under DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beijing_bmstc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/beijing_bmstc/beijing_bmstc_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.beijing_bmstc_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.beijing_bmstc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.beijing_bmstc_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320325902 is a Crossref-registered F4320* funder, so it MUST resolve to
exactly 1 row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320325902;

## Step 2: Create Beijing Municipal Science and Technology Commission Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.beijing_bmstc_awards
USING delta
AS
WITH
-- Path A: F4320325902 is F4320* (Crossref-registered) -> canonical metadata from the dim.
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320325902
),

awards_transformed AS (
    SELECT
        -- funder_award_id is always populated by the script (published number, or the
        -- stable synthetic key documented in the header), so the id hashes on it alone.
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        -- the script already converted 万元 -> CNY (x10,000)
        TRY_CAST(g.amount AS DOUBLE) as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'CNY' ELSE NULL END as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- BMSTC plan 课题 are commissioned R&D projects; the parent 项目 (programme) is the scheme.
        'research' as funding_type,

        NULLIF(TRIM(g.programme), '') as funder_scheme,

        'beijing_bmstc' as provenance,

        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.start_year AS INT)) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,

        -- Lead investigator: Chinese PI full name in family_name, given NULL (NSFC precedent).
        CASE
            WHEN (g.lead_family_name IS NOT NULL AND TRIM(g.lead_family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    NULLIF(TRIM(g.lead_family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.beijing_bmstc_raw g
    CROSS JOIN src_funder f
    WHERE g.display_name IS NOT NULL
      AND TRIM(g.display_name) != ''
      AND g.funder_award_id IS NOT NULL
      AND TRIM(g.funder_award_id) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beijing_bmstc' AND priority = 510;

-- Insert into openalex_awards_raw with priority.
-- Priority 510: direct-from-funder ingest. Higher wins under the
-- 2026-06-20 DESC dedup (oxjob #500), so it outranks acknowledgement shells
-- (priority 0) and grant-DOI stubs (priority 1).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    510 as priority
FROM openalex.awards.beijing_bmstc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_beijing_bmstc_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.beijing_bmstc_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency,
       start_date, end_date, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.beijing_bmstc_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt FROM openalex.awards.beijing_bmstc_awards
GROUP BY funding_type, funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.beijing_bmstc_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.3 / §6.7 coverage
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.beijing_bmstc_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'beijing_bmstc'
GROUP BY provenance, priority;